# Figure 3: MonkeyDrive validation analyses

Run the cells in order to reproduce Figure 3A (lateral motion) and Figure 3B (axial positioning), view the plots, and explore the result tables. See the [root README](../../README.md#interactive-notebooks) for setup and the [Figure 3 guide](README.md) for data and output details.

This notebook calls the existing analysis scripts using the notebook kernel's Python environment. It does not duplicate their statistical calculations. New outputs are saved under `figures/fig3/results/`; the supplied data and reference plots remain unchanged.

## Setup and input verification

The notebook works when its working directory is `figures/fig3/` or the repository root. Only Figure 3 inputs are needed; the large Figure 4 arrays are not required.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

# Find the repository from the current notebook directory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "verify_data.py").is_file() and (path / "figures" / "fig3").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the monkeydrive_paper checkout.")

FIG3 = ROOT / "figures" / "fig3"
OUTPUT_DIR = FIG3 / "results"
ENV = dict(os.environ, MPLBACKEND="Agg")

subprocess.run(
    [sys.executable, str(ROOT / "verify_data.py"), "--figure", "3"],
    cwd=ROOT, check=True,
)


def run_analysis(script, input_path, output_dir, *options):
    result = subprocess.run(
        [sys.executable, str(script), str(input_path),
         "--output-dir", str(output_dir), *map(str, options)],
        cwd=ROOT, env=ENV, capture_output=True, text=True,
    )
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)
    print(f"Saved outputs to {output_dir.relative_to(ROOT)}")

## Figure 3A: lateral motion

`rolling_window` controls smoothing of the displayed XY trajectories only. Leave it at **10** to reproduce the supplied default. Numerical analyses use unsmoothed valid measurements, so changing this setting does not change the statistics. Edit the value and rerun the following cells to explore the plot.

In [ ]:
rolling_window = 10
assert isinstance(rolling_window, int) and rolling_window >= 1

output_3a = OUTPUT_DIR / "fig3a"
run_analysis(
    FIG3 / "fig3a" / "lateral_motion_analysis.py",
    FIG3 / "fig3a" / "data",
    output_3a,
    "--rolling-window", rolling_window,
)
display(Image(filename=str(output_3a / "fig3a_lateral_motion_trajectories.png"), width=700))

### Summary statistics

RMS displacement columns are in micrometers. `raw` uses the original XY measurements; `detrended` removes linear trends; `stationary` describes the measurement floor.

In [ ]:
summary_3a = pd.read_csv(output_3a / "fig3a_lateral_motion_summary_statistics.csv")
display(summary_3a)

### Trial-level measurements and statistical comparisons

These DataFrames remain available for filtering, sorting, or selecting columns in additional cells.

In [ ]:
trials_3a = pd.read_csv(output_3a / "fig3a_lateral_motion_trial_metrics.csv")
tests_3a = pd.read_csv(output_3a / "fig3a_lateral_motion_statistical_tests.csv")
display(trials_3a)
display(tests_3a)

## Figure 3B: axial positioning

The analysis uses the measured displacements and the paper's **10-mm commanded distance**. The settings below change plot presentation only. Edit them and rerun this section to adjust the display.

In [ ]:
plot_width = 4.0
plot_height = 6.0
plot_font_size = 12.0
plot_dpi = 300
assert all(value > 0 for value in (plot_width, plot_height, plot_font_size, plot_dpi))

output_3b = OUTPUT_DIR / "fig3b"
run_analysis(
    FIG3 / "fig3b" / "axial_position.py",
    FIG3 / "fig3b" / "axial_position_data.csv",
    output_3b,
    "--plot-width", plot_width,
    "--plot-height", plot_height,
    "--plot-font-size", plot_font_size,
    "--plot-dpi", plot_dpi,
)
display(Image(filename=str(output_3b / "fig3b_signed_positioning_error.png"), width=450))

### Descriptive statistics

Displacement and error columns are in millimeters. Signed error is measured minus commanded displacement; negative values indicate undershoot.

In [ ]:
summary_3b = pd.read_csv(output_3b / "fig3b_axial_positioning_descriptive_statistics.csv")
display(summary_3b)

### Accuracy comparisons

Welch's ANOVA tests differences in absolute positioning error. Games–Howell comparisons assess the pairs of drives.

In [ ]:
welch_3b = pd.read_csv(output_3b / "fig3b_axial_positioning_welch_anova.csv")
games_howell_3b = pd.read_csv(output_3b / "fig3b_axial_positioning_games_howell_pairwise.csv")
display(welch_3b)
display(games_howell_3b)

### Precision comparisons

The Brown–Forsythe tests compare variability. Pairwise p-values include Holm adjustment.

In [ ]:
brown_forsythe_3b = pd.read_csv(output_3b / "fig3b_axial_positioning_brown_forsythe_omnibus.csv")
pairwise_3b = pd.read_csv(output_3b / "fig3b_axial_positioning_brown_forsythe_pairwise_holm.csv")
display(brown_forsythe_3b)
display(pairwise_3b)

## Saved results

Each panel's output directory contains PNG/PDF plots, CSV tables, and a text report. Rerunning a panel replaces its generated outputs. Restore the default settings above and rerun all cells to return to the supplied plotting defaults. Final manuscript styling may differ from these standalone plots.

## Relationship to the manuscript

These plots reproduce the Figure 3 analyses, but their formatting differs from the manuscript's final figure. Axis presentation, colors, labels, and layout may differ; the underlying data and calculations are unchanged.